LLM - huggingface LLM (default : gpt-3.5-turbo)
https://docs.llamaindex.ai/en/stable/module_guides/models/llms/usage_custom/

EMBEDDING MODEL - BAAI (default : text-embedding-ada-002)
https://docs.llamaindex.ai/en/stable/module_guides/models/embeddings/

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

# set the batch size (default : 10)
embed_model = OpenAIEmbedding(embed_batch_size=42)

Settings.embed_model = OpenAIEmbedding()

VECTOR STORE - 수정 X (customize하려면 pinecone 써야함)

In [2]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)

QUERY ENGINE

In [3]:
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine

# build index
index = VectorStoreIndex.from_documents(documents)

# configure retriever
retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=2,
)

# configure response synthesizer
response_synthesizer = get_response_synthesizer(
    response_mode="tree_summarize",
)

# assemble query engine
query_engine = RetrieverQueryEngine(
    retriever=retriever,
    response_synthesizer=response_synthesizer,
)

EVALUATION

In [4]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [5]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [6]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [7]:
responses_str = []
responses = []
count=0

In [8]:
for question in questions:
    count+=1
    query = f"{question}."
    response = query_engine.query(query)
    responses.append(response)
    
    response_str=str(response)

    print(count, response)
    if "True" in response_str:
        response_str="True"
    elif "False" in response_str:
        response_str="False"
    else:
        print("error")
        
    responses_str.append(response_str)

1 True
2 False
3 True
4 False
5 True
6 True
7 True
8 True
9 False
10 True
11 True
12 True
13 False
14 True
15 False
16 True
17 False
18 True
19 False
20 True
21 False
22 True
23 True
24 False
25 True
26 False
27 True
28 False
29 True
30 True
31 False
32 True
33 True
34 False.
35 True
36 True
37 False
38 False
39 True
40 False
41 True
42 False
43 False
44 True
45 True
46 False
47 True
48 False
49 True
50 False
51 True
52 False
53 False
54 True
55 True
56 True
57 False
58 True
59 False
60 True
61 True
62 True
63 True
64 True
65 True
66 True
67 True
68 True
69 True
70 True
71 True
72 True
73 True
74 True
75 True


In [9]:
correct_count=0
number=0

for response, answer, response_str, question in zip(responses, answers, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 11 Without fourth-dimensional gateways, scientists would still be able to alter the fabric of space-time. (True/False)
RESPONSE True
CORRECT ANSWER False

<<wrong>>
 60 The identification of a fourth spatial dimension has enabled the development of a breathable liquid atmosphere on Earth. (True/False)
RESPONSE True
CORRECT ANSWER False

<<wrong>>
 71 Aquatic organisms need extensive changes to thrive in the newly created liquid atmosphere. (True/False)
RESPONSE True
CORRECT ANSWER False

<<wrong>>
 74 The space station derives its energy from the neutron star using advanced fusion technology. (True/False)
RESPONSE True
CORRECT ANSWER False

correct_count: 71


In [10]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

Total Questions: 75
Correct Answers: 71
Accuracy: 94.67%
